# Training a Bert from Scratch

In [12]:
from transformers import BertConfig, BertForMaskedLM, BertTokenizerFast, Trainer, TrainingArguments, DataCollatorForLanguageModeling,pipeline, AutoTokenizer,AutoModel
from sentence_transformers import SentenceTransformer
import torch
from datasets import load_dataset
import gc

In [2]:
device ='cuda'

In [3]:
def del_vars(variable_names=[]):
    for name in variable_names:
        try:
            del globals()[name]
        except KeyError:
            pass  # ignore variables that have already been deleted
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()

In [4]:
#Tokenizer
bert_tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")
config = BertConfig(
    vocab_size=bert_tokenizer.vocab_size, hidden_size = 128, num_hidden_layers=2, num_attention_heads=4, intermediate_size=512, max_position_embeddings=128
)
bert = BertForMaskedLM(config)

In [5]:
def tokenize(example, tokenizer=bert_tokenizer):
    return tokenizer(example["text"], truncation=True, max_length=128, padding="max_length")

mlm_dataset = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train")
print(mlm_dataset)
mlm_dataset = mlm_dataset.map(tokenize, batched=True)

Dataset({
    features: ['text'],
    num_rows: 36718
})


In [6]:
args = TrainingArguments(output_dir="./my_bert", num_train_epochs=10, per_device_train_batch_size=16)
mlm_collator = DataCollatorForLanguageModeling(bert_tokenizer, mlm=True, mlm_probability=0.15)
trainer = Trainer(model=bert, args=args, train_dataset=mlm_dataset, data_collator=mlm_collator)
trainer_output = trainer.train()

Step,Training Loss
500,8.915712
1000,7.498797
1500,7.313021
2000,7.218079
2500,7.164017
3000,7.124387
3500,7.116259
4000,7.054048
4500,7.061555
5000,6.995123


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [7]:
torch.manual_seed(42)
fill_mask = pipeline("fill-mask", model=bert, tokenizer=bert_tokenizer)
top_predictions= fill_mask("The capital of [MASK] is Rome.")
top_predictions[0]

{'score': 0.031029047444462776,
 'token': 1010,
 'token_str': ',',
 'sequence': 'the capital of, is rome.'}

In [8]:
del_vars(["bert_tokenizer", "config", "bert", "mlm_dataset", "args",
          "mlm_collator", "trainer", "trainer_output", "fill_mask",
          "top_predictions"])

In [11]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")
inputs = tokenizer(["I like soccer", "Hello, World!"],
                   padding=True, return_tensors="pt")
outputs = model(**inputs)
cls_embedding = outputs.last_hidden_state[:, 0, :]
cls_embedding.shape

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


torch.Size([2, 768])

In [13]:
model = SentenceTransformer("all-MiniLM-L6-v2")
sentences = ["She's shopping", 'She bought some shoes', "She's working"]
embeddings = model.encode(sentences, convert_to_tensor=True)
similarities = model.similarity(embeddings, embeddings)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]